In [ ]:
!pip install clearml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.3 MB/s eta 0:00:00


# Pre training on EN data

In [ ]:
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from clearml import Task

In [ ]:
task = Task.init(
    project_name="cross-lingual-lm",
    task_name="english_pretraining_roberta_1M_tokens",
    task_type=Task.TaskTypes.training
)

logger = task.get_logger()


ClearML Task: created new task id=d58c1cf9dc304b119b606b52d07b4443


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


2026-04-20 09:02:21,274 - clearml.Task - INFO - Storing jupyter notebook directly as code
ClearML results page: https://app.clear.ml/projects/9bf855ce67034d8d9b44bf6ae4fbf457/experiments/d58c1cf9dc304b119b606b52d07b4443/output/log


In [ ]:
set_seed(42)

MODEL_NAME = "roberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 8
GRAD_ACCUM = 4
EPOCHS = 1
LR = 5e-5
SAMPLE_SIZE = 100000
OUTPUT_DIR = "./models/english_lm_roberta"


In [ ]:
task.connect({
    "model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "epochs": EPOCHS,
    "lr": LR,
    "sample_size": SAMPLE_SIZE
})

{'model': 'roberta-base',
 'max_length': 128,
 'batch_size': 8,
 'grad_accum': 4,
 'epochs': 1,
 'lr': 5e-05,
 'sample_size': 100000}

In [ ]:
dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

train_dataset = dataset["train"]

train_dataset = train_dataset.filter(lambda x: len(x["text"].strip()) > 0)

train_dataset = train_dataset.select(range(SAMPLE_SIZE))

print(f"Train samples: {len(train_dataset)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Train samples: 100000


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_dataset = train_dataset.map(
    tokenize,
    batched=True,
    num_proc=os.cpu_count(),
    remove_columns=["text"]
)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map (num_proc=2):   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
import torch

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    num_train_epochs=EPOCHS,
    learning_rate=LR,

    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,

    fp16=True,

    dataloader_num_workers=2,

    report_to=["none"],
    run_name="roberta_en_pretrain_10M",

    optim="adamw_torch",
    lr_scheduler_type="linear",
    warmup_ratio=0.1
)
torch.manual_seed(42)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

logger.report_text("English pretraining finished")


Step,Training Loss
100,8.151934
200,6.878245
300,6.844953
400,6.889366
500,6.771423
600,6.673158
700,6.699997
800,6.551566
900,6.625379
1000,6.608249


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-20 09:13:57,705 - clearml.frameworks - INFO - Found existing registered model id=10f1619f540b4a17be88dcfdce8e4a62 [/content/models/english_lm_roberta/checkpoint-1000/training_args.bin] reusing it.
2026-04-20 09:14:14,382 - clearml.frameworks - INFO - Found existing registered model id=73e680ed4f2a44b0be7a25ceb1e550ab [/content/models/english_lm_roberta/checkpoint-1000/optimizer.pt] reusing it.
2026-04-20 09:14:20,157 - clearml.frameworks - INFO - Found existing registered model id=c447947f09714f198fea8acd502d40fb [/content/models/english_lm_roberta/checkpoint-1000/scheduler.pt] reusing it.
2026-04-20 09:14:25,878 - clearml.frameworks - INFO - Found existing registered model id=264d395972bc4a7987327474507edab3 [/content/models/english_lm_roberta/checkpoint-1000/scaler.pt] reusing it.
2026-04-20 09:14:31,744 - clearml.frameworks - INFO - Found existing registered model id=f839d01b59c246289251fb3380fc1766 [/content/models/english_lm_roberta/checkpoint-1000/rng_state.pth] reusing i

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-20 09:21:44,205 - clearml.frameworks - INFO - Found existing registered model id=0a295e54187242a28f82edbae7516446 [/content/models/english_lm_roberta/checkpoint-2000/training_args.bin] reusing it.
2026-04-20 09:21:54,926 - clearml.frameworks - INFO - Found existing registered model id=cc1d9fb7d02345a48c0db6154d04ac85 [/content/models/english_lm_roberta/checkpoint-2000/optimizer.pt] reusing it.
2026-04-20 09:22:00,598 - clearml.frameworks - INFO - Found existing registered model id=c33ebbf799b945eb971d4d4e722040aa [/content/models/english_lm_roberta/checkpoint-2000/scheduler.pt] reusing it.
2026-04-20 09:22:06,455 - clearml.frameworks - INFO - Found existing registered model id=9e5a81cc22af4b8987784af8efe95134 [/content/models/english_lm_roberta/checkpoint-2000/scaler.pt] reusing it.
2026-04-20 09:22:12,111 - clearml.frameworks - INFO - Found existing registered model id=a578799377174649968d24c11623a2ff [/content/models/english_lm_roberta/checkpoint-2000/rng_state.pth] reusing i

...

2026-04-20 09:29:33,261 - clearml.frameworks - INFO - Found existing registered model id=e182e87bd2634410aefe67d620a66d6c [/content/models/english_lm_roberta/checkpoint-3000/training_args.bin] reusing it.
2026-04-20 09:30:10,047 - clearml.frameworks - INFO - Found existing registered model id=8d0e45f83ba14317b97b7782c81d8c79 [/content/models/english_lm_roberta/checkpoint-3000/optimizer.pt] reusing it.
2026-04-20 09:30:15,790 - clearml.frameworks - INFO - Found existing registered model id=495b9804101a40ec993342c85abb9e88 [/content/models/english_lm_roberta/checkpoint-3000/scheduler.pt] reusing it.
2026-04-20 09:30:21,400 - clearml.frameworks - INFO - Found existing registered model id=919737c1dc22439fa6128ced83329286 [/content/models/english_lm_roberta/checkpoint-3000/scaler.pt] reusing it.
2026-04-20 09:30:27,101 - clearml.frameworks - INFO - Found existing registered model id=4564a6bea9c44dcfafcd32fc0bd0ecfc [/content/models/english_lm_roberta/checkpoint-3000/rng_state.pth] reusing i

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-20 09:32:10,991 - clearml.frameworks - INFO - Found existing registered model id=102ee4207dfe48448e78c55b043b545b [/content/models/english_lm_roberta/checkpoint-3125/training_args.bin] reusing it.
2026-04-20 09:33:07,204 - clearml.frameworks - INFO - Found existing registered model id=3201475806d4498599026faf217254e1 [/content/models/english_lm_roberta/checkpoint-3125/optimizer.pt] reusing it.
2026-04-20 09:33:13,717 - clearml.frameworks - INFO - Found existing registered model id=a0705b75374b427ba33f48ea0efc80a8 [/content/models/english_lm_roberta/checkpoint-3125/scheduler.pt] reusing it.
2026-04-20 09:33:21,093 - clearml.frameworks - INFO - Found existing registered model id=7799f965e6404cefab00d2b2a8578fea [/content/models/english_lm_roberta/checkpoint-3125/scaler.pt] reusing it.
2026-04-20 09:33:28,340 - clearml.frameworks - INFO - Found existing registered model id=92d28d063ece4c16833ad2879a1c640a [/content/models/english_lm_roberta/checkpoint-3125/rng_state.pth] reusing i

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-20 09:33:58,244 - clearml.frameworks - INFO - Found existing registered model id=7cbc22c21c2744119e04f950b1f4677e [/content/models/english_lm_roberta/training_args.bin] reusing it.
English pretraining finished


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


# Evaluate on EN PPL

In [ ]:
import math
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

In [ ]:
DATASET_NAME = "wikitext"
DATASET_CONFIG = "wikitext-103-raw-v1"

MAX_LENGTH = 128
BATCH_SIZE = 8
EVAL_SAMPLES = 3000

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split="test")

dataset = dataset.filter(lambda x: x["text"] and len(x["text"].strip()) > 0)
dataset = dataset.select(range(min(EVAL_SAMPLES, len(dataset))))


Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/nlp_project/models/english_lm_roberta"
MODEL_PATH = "/content/models/english_lm_roberta/checkpoint-3125"

In [ ]:
def eval_ppl(MODEL_PATH, dataset):
    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH
        )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)
    model.to(device)
    model.eval()


    tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])

    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    args = TrainingArguments(
        output_dir="./eval_en_tmp",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to=[]
    )

    trainer = Trainer(
        model=model,
        args=args,
        eval_dataset=tokenized_dataset,
        data_collator=collator
    )


    torch.manual_seed(42)
    metrics = trainer.evaluate()

    loss = metrics["eval_loss"]
    perplexity = math.exp(loss)

    print(f"Loss: {loss:.4f}")
    print(f"Perplexity: {perplexity:.2f}")

In [ ]:
eval_ppl(MODEL_PATH, dataset)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Map:   0%|          | 0/2891 [00:00<?, ? examples/s]

Loss: 1.4544
Perplexity: 4.28


In [ ]:
eval_ppl("roberta-base", dataset)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/2891 [00:00<?, ? examples/s]

Loss: 2.4993
Perplexity: 12.17


# Evaluate on Swahili PPL

In [ ]:
dataset = load_dataset("ngusadeep/Swahili-Corpus-Dataset")["train"]

# Remove empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

# Limit dataset size (low-resource simulation)
dataset_dict = dataset.train_test_split(test_size=0.1, seed=42, shuffle=True)

raw_train_data = dataset_dict["train"]
eval_data = dataset_dict["test"]

eval_data = eval_data.select(range(min(3000, len(eval_data))))

print(f"Eval size:  {len(eval_data)}")

README.md: 0.00B [00:00, ?B/s]

Swahili_Corpus_combined.txt:   0%|          | 0.00/253M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1693227 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1693227 [00:00<?, ? examples/s]

Eval size:  3000


In [ ]:
eval_ppl(MODEL_PATH, eval_data)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Loss: 4.5371
Perplexity: 93.41


In [ ]:
eval_ppl("roberta-base", eval_data)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Loss: 4.2405
Perplexity: 69.44
